In [ ]:
# ── 패키지 설치 (처음 한 번만 실행) ──────────────────────
!pip install kiwipiepy pandas tqdm

In [ ]:
INPUT_FILE  = "reviews.txt"      # ← 리뷰 txt 파일 경로
OUTPUT_DIR  = "./output"         # ← 결과물 저장 폴더 (자동 생성됨)

# 분석 파라미터 (기본값 그대로 써도 됩니다)
WINDOW_SIZE = 5      # collocation 윈도우 (좌우 각 5 토큰)
MIN_FREQ    = 3      # 최소 빈도 (3번 미만 등장 단어 제외)
TOP_N       = 50     # 공기어 수

print("✅ 설정 완료")

In [ ]:
# ── 라이브러리 불러오기 ──────────────────────────────────
import math, json, re
from collections import Counter
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
from kiwipiepy import Kiwi

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("✅ 라이브러리 로드 완료")

In [ ]:
# ── 데이터 로드 ──────────────────────────────────────────
with open(INPUT_FILE, encoding="utf-8") as f:
    reviews = [line.strip() for line in f if line.strip()]

print(f"총 리뷰 수: {len(reviews):,}개")
print("\n--- 처음 3개 미리보기 ---")
for i, r in enumerate(reviews[:3]):
    print(f"[{i+1}] {r[:80]}...")

In [ ]:
# ── 형태소 분석 (아버지도/아버지는 → 아버지 자동 통합) ──

# 분석 대상 단어
TARGET_LEMMAS = {"아버지", "아빠"}

# 제외할 품사 (조사, 어미, 구두점 등)
STOPPOS = {
    "JKS","JKC","JKG","JKO","JKB","JKV","JKQ",  # 격조사
    "JX","JC",                                      # 보조사, 접속조사
    "EP","EF","EC","ETN","ETM",                    # 어미
    "XSV","XSA","XSN","XPN",                       # 접사
    "SF","SP","SS","SE","SO","SW",                  # 구두점/기호
    "SL","SH",                                      # 외국어, 한자
}

print("kiwi 초기화 중... (약 10초 소요)")
kiwi = Kiwi()

tokenized = []  # 각 리뷰의 (표제어, 품사) 리스트

for review in tqdm(reviews, desc="형태소 분석"):
    tokens = []
    result = kiwi.analyze(review)
    if not result:
        tokenized.append([])
        continue
    for token in result[0].tokens:
        lemma = token.form
        pos   = token.tag.name if hasattr(token.tag, 'name') else str(token.tag)
        if pos in STOPPOS:
            continue
        if len(lemma) < 2 and lemma not in TARGET_LEMMAS:
            continue
        tokens.append((lemma, pos))
    tokenized.append(tokens)

all_tokens = [t[0] for toks in tokenized for t in toks]
print(f"\n✅ 형태소 분석 완료")
print(f"   전체 토큰 수: {len(all_tokens):,}개")
print(f"   '아버지' 등장: {all_tokens.count('아버지'):,}회")
print(f"   '아빠' 등장:   {all_tokens.count('아빠'):,}회")

In [ ]:
# ── Collocation 통계 계산 (MI / T-score / log-likelihood) 

N        = len(all_tokens)
freq_all = Counter(all_tokens)
cooc     = Counter()
target_total = 0

for tokens in tokenized:
    lemmas = [t[0] for t in tokens]
    for i, lemma in enumerate(lemmas):
        if lemma in TARGET_LEMMAS:
            target_total += 1
            start = max(0, i - WINDOW_SIZE)
            end   = min(len(lemmas), i + WINDOW_SIZE + 1)
            for j in range(start, end):
                if j != i:
                    cooc[lemmas[j]] += 1

rows = []
for collocate, O11 in cooc.items():
    if O11 < MIN_FREQ or collocate in TARGET_LEMMAS:
        continue
    f_t = target_total
    f_c = freq_all[collocate]
    E11 = max((f_t * f_c) / N, 1e-9)
    mi  = math.log2(O11 / E11)
    t_score = (O11 - E11) / math.sqrt(O11)

    def sll(o, e): return o * math.log(o/e) if o > 0 and e > 0 else 0
    E12 = max((f_t * (N - f_c)) / N, 1e-9)
    E21 = max(((N - f_t) * f_c) / N, 1e-9)
    E22 = max(((N - f_t) * (N - f_c)) / N, 1e-9)
    O12, O21, O22 = f_t - O11, f_c - O11, N - O11 - (f_t-O11) - (f_c-O11)
    ll = 2 * (sll(O11,E11) + sll(O12,E12) + sll(O21,E21) + sll(O22,E22))

    rows.append({"collocate":collocate, "빈도(O11)":O11,
                 "MI":round(mi,4), "T_score":round(t_score,4),
                 "log_likelihood":round(ll,4)})

cooc_df = pd.DataFrame(rows).sort_values("log_likelihood", ascending=False)
cooc_df.to_csv(f"{OUTPUT_DIR}/collocation_results.csv", index=False, encoding="utf-8-sig")

print(f"✅ Collocation 완료 (공기어 {len(cooc_df):,}개)")
print(f"   저장 위치: {OUTPUT_DIR}/collocation_results.csv\n")
cooc_df.head(20)